# FINAL Kaggle T4×2 Public-Paper Run — v2 FIXED

**Use a fresh Kaggle session. Do not run the older notebook first.**  
This v2 fixes the NumPy/SciPy ABI crash by **not upgrading Kaggle's numerical stack**. It installs only the Hugging Face packages required for the experiment.

# FINAL PUBLIC-PAPER RUN — Transformer MLP Transplants
## Kaggle **2×T4** only — alternative controls + Receiver-Conditioned Causal Transformation (RCCT)

**Run this notebook instead of the older final control notebooks.** It subsumes the remaining public-paper work, so running the older notebook first would mostly waste GPU.

The accepted paper already establishes the main crossed-receiver result at much larger scale. This notebook adds only what is still useful:

1. **Alternative predictor baselines** on a fresh, deterministic control panel:
   - displacement norm relative to receiver state, `||Δ||/||r||`;
   - exact **Attribution Patching** baselines for native top-token log-probability and top1–top2 margin;
   - matched local-response magnitude `qp2`;
   - physical equal-norm transplants.
2. **Numerical closure**:
   - Qwen/Mistral: epsilon-halving + Richardson finite-difference audit;
   - six Pythia-70M runs: full-FP32 **exact JVP vs centered finite difference**.
3. **New scientific test — RCCT**:
   - keep the *same natural displacement* `Δ = u_j - u_i`;
   - measure its full causal effect vector after every downstream layer in its original host and in alternative receivers;
   - separate **fidelity**, **gain**, and **rewrite residual**;
   - test whether a receiver-specific transformation learned from training displacements predicts **held-out, unseen displacements** better than identity, scalar gain, a global map, and a wrong-receiver map.

### Frozen hypothesis
> **Receiver state may impose a reproducible transformation on the causal effects of unseen natural MLP displacements.**

A negative result is still interpretable: if receiver-specific maps fail on held-out displacements, then the interaction is genuinely pair-specific rather than compressible into a reusable receiver rule.

### Important terminology
`RCCT` is a working label for the experiment. Do **not** claim a new Fisher geometry. Fisher remains an established local sensitivity diagnostic; the new target is the **full causal effect trajectory**, measured by physical forward interventions.

In [ ]:
# IMPORTANT: run this notebook in a FRESH Kaggle T4 x2 session.
# The previous notebook upgraded NumPy/SciPy inside the live kernel, which can break
# compiled SciPy extensions until a restart. This cell deliberately leaves Kaggle's
# preinstalled numerical stack untouched.
#
# We install ONLY the Hugging Face packages needed for Qwen3/Mistral/C4 loading.
%pip -q install --upgrade-strategy only-if-needed \
  "transformers>=4.56,<5" \
  "datasets>=3,<5" \
  "accelerate>=1.2,<2" \
  "safetensors>=0.4.5"

print("Minimal HF install complete. NumPy/SciPy/Pandas/sklearn were NOT modified.")


In [ ]:
import os, gc, json, math, random, hashlib, traceback, zipfile, shutil, warnings, time
from pathlib import Path

# Numerical stack: use Kaggle's preinstalled binaries. Do NOT upgrade these in-session.
try:
    import numpy as np
    import pandas as pd
    from scipy.stats import spearmanr, pearsonr, rankdata
    from sklearn.decomposition import PCA
    from sklearn.linear_model import Ridge
    import matplotlib.pyplot as plt
except Exception as e:
    raise RuntimeError(
        "Kaggle numerical stack is not clean. This usually means NumPy/SciPy were upgraded "
        "earlier in the SAME session. Use Session -> Restart Session (or start a fresh T4 x2 "
        "session), then Run All with THIS v2 notebook. Original error: " + repr(e)
    )

from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import requests

warnings.filterwarnings('ignore')

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
try:
    import scipy
    print("scipy:", scipy.__version__)
except Exception:
    pass
try:
    import sklearn
    print("sklearn:", sklearn.__version__)
except Exception:
    pass
print("torch:", torch.__version__)

print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, torch.cuda.get_device_name(i), round(p.total_memory/2**30, 1), "GiB")

assert torch.cuda.device_count() >= 2, "Select Kaggle Accelerator: GPU T4 x2."
assert all("T4" in torch.cuda.get_device_name(i).upper() for i in [0, 1]), \
    "This notebook is frozen for Kaggle T4 x2."

# Sanity test the exact imports that failed in the previous run.
_x = np.asarray([3.0, 1.0, 2.0])
assert np.all(np.isfinite(_x))
assert np.isfinite(spearmanr([1,2,3], [1,2,3]).statistic)
print("Preflight PASSED: numerical stack + SciPy + sklearn + T4 x2 are healthy.")


## 0. Frozen configuration

The new RCCT study needs **more distinct displacements than receivers**, because the strongest test is held-out generalization across incoming computations.

- `D=96` natural displacements;
- `R=8` alternative receivers;
- 72 train / 24 held-out displacements for transformation maps;
- primary Qwen layer 14 and Mistral layer 12, matching the accepted paper;
- pair batch 4, matching the proven T4×2 workflow.

If you have the exact accepted-paper context manifest, set `CONTEXT_MANIFEST`. Otherwise this notebook creates a **fresh deterministic C4 validation control panel** and labels it as such.

In [ ]:
SEED = 26091016
SEQ_LEN = 64
D = 96
R = 8
PAIR_BATCH = 4
FD_EPS = 0.075
RICHARDSON_N = 16
MAP_RANK = 16
RIDGE_ALPHA = 1.0
TRAIN_FRAC = 0.75

MODEL_SPECS = {
    'qwen3_8b': {'model_id':'Qwen/Qwen3-8B','layer':14},
    'mistral_7b': {'model_id':'mistralai/Mistral-7B-v0.3','layer':12},
}

PYTHIA_MODELS = {
    'base':'EleutherAI/pythia-70m',
    'seed1':'EleutherAI/pythia-70m-seed1',
    'seed2':'EleutherAI/pythia-70m-seed2',
    'seed3':'EleutherAI/pythia-70m-seed3',
    'seed4':'EleutherAI/pythia-70m-seed4',
    'seed5':'EleutherAI/pythia-70m-seed5',
}
PYTHIA_LAYER = 2
PYTHIA_PAIRS_PER_RUN = 24

# Optional: '/kaggle/input/.../contexts.jsonl'
CONTEXT_MANIFEST = None

# Optional phone push via ntfy; leave blank to disable.
NTFY_TOPIC = ''

OUTDIR = Path('/kaggle/working/mlp_final_rcct')
OUTDIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

CONFIG = dict(seed=SEED,seq_len=SEQ_LEN,D=D,R=R,pair_batch=PAIR_BATCH,fd_eps=FD_EPS,
              map_rank=MAP_RANK,ridge_alpha=RIDGE_ALPHA,train_frac=TRAIN_FRAC,
              models=MODEL_SPECS,pythia_models=PYTHIA_MODELS,
              pythia_layer=PYTHIA_LAYER,pythia_pairs_per_run=PYTHIA_PAIRS_PER_RUN)
(OUTDIR/'frozen_config.json').write_text(json.dumps(CONFIG,indent=2))
print(json.dumps(CONFIG,indent=2))

In [ ]:
def notify(title, body, priority='default'):
    if not NTFY_TOPIC.strip(): return
    try:
        requests.post(f'https://ntfy.sh/{NTFY_TOPIC.strip()}', data=body.encode(),
                      headers={'Title':title,'Priority':priority}, timeout=15).raise_for_status()
    except Exception as e:
        print('ntfy warning:',repr(e))

def sha256_text(s): return hashlib.sha256(s.encode('utf-8',errors='ignore')).hexdigest()

def atomic_csv(df,path):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False); os.replace(tmp,path)

def cleanup(model=None):
    if model is not None: del model
    gc.collect(); torch.cuda.empty_cache()
    try: torch.cuda.ipc_collect()
    except: pass

def load_contexts(path=None,n=240):
    if path and Path(path).exists():
        p=Path(path)
        if p.suffix.lower()=='.jsonl': rows=[json.loads(x) for x in p.read_text().splitlines() if x.strip()]
        elif p.suffix.lower()=='.json': rows=json.loads(p.read_text())
        else: rows=[{'text':x.strip()} for x in p.read_text().splitlines() if x.strip()]
        out=[]
        for r in rows:
            text=r['text'] if isinstance(r,dict) else str(r)
            cid=r.get('context_id',sha256_text(text)[:20]) if isinstance(r,dict) else sha256_text(text)[:20]
            out.append({'context_id':cid,'text':text})
        print('Loaded supplied context manifest:',len(out))
        return out[:n]

    print('No manifest supplied: creating a fresh deterministic C4 validation control panel.')
    ds=load_dataset('allenai/c4','en',split='validation',streaming=True)
    out=[]
    for ex in ds:
        text=(ex.get('text') or '').strip()
        if len(text)<500: continue
        h=sha256_text(text)
        if int(h[:8],16)%1000 >= 80: continue
        out.append({'context_id':h[:20],'text':text})
        if len(out)>=n: break
    print('C4 contexts:',len(out))
    return out

RAW_CONTEXTS=load_contexts(CONTEXT_MANIFEST,n=max(2*D+R+40,120))

## 1. Core intervention helpers

For a natural host–donor pair,

\[
\Delta_d=u_j-u_i.
\]

At the original host and at receiver `k`, the **same** `Δ_d` is added to the final-token MLP output. At every downstream layer `m` we measure the physical causal effect:

\[
c^{(m)}_{d,i}=h_m(i;+\Delta_d)-h_m(i),\qquad
c^{(m)}_{d,k}=h_m(k;+\Delta_d)-h_m(k).
\]

At the injection layer the causal effects should start almost identical. Any later divergence is created by the different receiving states / suffix trajectories.

In [ ]:
def get_layers(model):
    if hasattr(model,'model') and hasattr(model.model,'layers'): return model.model.layers
    if hasattr(model,'gpt_neox') and hasattr(model.gpt_neox,'layers'): return model.gpt_neox.layers
    if hasattr(model,'transformer') and hasattr(model.transformer,'h'): return model.transformer.h
    raise AttributeError('Unsupported model layer layout')

def get_mlp(model,idx): return get_layers(model)[idx].mlp

def input_device(model): return model.get_input_embeddings().weight.device

def encode_exact(tok,text,n=SEQ_LEN):
    ids=tok(text,add_special_tokens=False,truncation=False)['input_ids']
    if len(ids)<n: return None
    return torch.tensor(ids[:n],dtype=torch.long)

def tokenized_rows(tok,raw,need):
    rows=[]
    for r in raw:
        ids=encode_exact(tok,r['text'])
        if ids is None: continue
        rows.append({**r,'input_ids':ids})
        if len(rows)>=need: break
    if len(rows)<need: raise RuntimeError(f'Need {need} tokenizable contexts, got {len(rows)}')
    return rows

def kl_native_patch(z0,z1):
    lp=torch.log_softmax(z0.float(),dim=-1); lq=torch.log_softmax(z1.float(),dim=-1); p=lp.exp()
    return (p*(lp-lq)).sum(dim=-1)

def fisher_energy(v,p):
    pv=(p*v).sum(dim=-1)
    return .5*((p*v.square()).sum(dim=-1)-pv.square())

def qp2(v): return .5*v.square().sum(dim=-1)

def centered(x): return x-x.mean(dim=-1,keepdim=True)

def cosine_rows(a,b,eps=1e-12):
    return (a*b).sum(-1)/(a.norm(dim=-1)*b.norm(dim=-1)).clamp_min(eps)

def causal_metrics(ref,effect):
    # shapes (...,H)
    dot=(ref*effect).sum(-1); ref2=ref.square().sum(-1).clamp_min(1e-12)
    gain=dot/ref2
    resid=effect-gain[...,None]*ref
    fidelity=cosine_rows(ref,effect)
    rewrite=resid.norm(dim=-1)/effect.norm(dim=-1).clamp_min(1e-12)
    return fidelity,gain,rewrite

def load_large(spec):
    tok=AutoTokenizer.from_pretrained(spec['model_id'],use_fast=True)
    if tok.pad_token_id is None: tok.pad_token_id=tok.eos_token_id
    model=AutoModelForCausalLM.from_pretrained(
        spec['model_id'],torch_dtype=torch.float16,device_map='balanced',
        max_memory={0:'12000MiB',1:'12000MiB','cpu':'24GiB'},
        low_cpu_mem_usage=True,attn_implementation='sdpa')
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    return model,tok

@torch.inference_mode()
def native_forward(model,ids_cpu,layer_idx,capture_mlp=True,capture_traj=True):
    buf=[]
    h=None
    if capture_mlp:
        def hook(module,inp,out): buf.append(out[:,-1,:].detach().float().cpu())
        h=get_mlp(model,layer_idx).register_forward_hook(hook)
    try:
        out=model(input_ids=ids_cpu.to(input_device(model)),use_cache=False,
                  output_hidden_states=capture_traj,return_dict=True)
        logits=out.logits[:,-1,:].detach().float().cpu()
        traj=None
        if capture_traj:
            # hidden_states[0]=embeddings; [layer_idx+1] is output of target decoder layer
            traj=torch.stack([x[:,-1,:].detach().to('cpu',dtype=torch.float16)
                              for x in out.hidden_states[layer_idx+1:]],dim=1)
        u=buf[-1] if capture_mlp else None
        return u,traj,logits
    finally:
        if h is not None: h.remove()

@torch.inference_mode()
def patched_forward(model,ids_cpu,layer_idx,deltas_cpu,alpha=1.0,capture_traj=False):
    B=deltas_cpu.shape[0]
    if ids_cpu.ndim==1: ids_cpu=ids_cpu[None,:].repeat(B,1)
    assert ids_cpu.shape[0]==B
    mlp=get_mlp(model,layer_idx)
    def hook(module,inp,out):
        d=deltas_cpu.to(out.device,dtype=out.dtype)
        add=torch.zeros_like(out); add[:,-1,:]=d
        return out+float(alpha)*add
    h=mlp.register_forward_hook(hook)
    try:
        out=model(input_ids=ids_cpu.to(input_device(model)),use_cache=False,
                  output_hidden_states=capture_traj,return_dict=True)
        logits=out.logits[:,-1,:].detach().float().cpu()
        traj=None
        if capture_traj:
            traj=torch.stack([x[:,-1,:].detach().to('cpu',dtype=torch.float16)
                              for x in out.hidden_states[layer_idx+1:]],dim=1)
        return traj,logits
    finally:
        h.remove()

## 2. Exact Attribution-Patching baselines

For each receiver we perform one native suffix-backpropagation from the target MLP output and obtain:

- `AtP_toplogp = |∇_u log p(y*) · Δ|`
- `AtP_margin = |∇_u (z_top1-z_top2) · Δ|`

This is a direct gradient-based Attribution Patching baseline. We do **not** call the first derivative of KL an AtP baseline, because KL is stationary at the native point and that derivative is zero.

In [ ]:
def attribution_gradients(model,receiver_ids,layer_idx):
    captured={}
    mlp=get_mlp(model,layer_idx)
    def hook(module,inp,out):
        # Cut the graph before the intervention site; build grad graph only through suffix.
        x=out.detach().requires_grad_(True)
        captured['u']=x
        return x
    h=mlp.register_forward_hook(hook)
    try:
        with torch.enable_grad():
            out=model(input_ids=receiver_ids[None,:].to(input_device(model)),use_cache=False,return_dict=True)
            z=out.logits[0,-1,:]
            lp=torch.log_softmax(z.float(),dim=-1)
            top2=z.detach().float().topk(2).indices
            score_top=lp[top2[0]]
            score_margin=z[top2[0]]-z[top2[1]]
            g1=torch.autograd.grad(score_top,captured['u'],retain_graph=True,create_graph=False)[0][0,-1,:]
            g2=torch.autograd.grad(score_margin,captured['u'],retain_graph=False,create_graph=False)[0][0,-1,:]
        return g1.detach().float().cpu(),g2.detach().float().cpu()
    finally:
        h.remove(); gc.collect(); torch.cuda.empty_cache()

## 3. Run one model: cache natural states, build RCCT trajectories, and compute alternative controls

For every receiver/displacement pair we save:

- full-dose KL;
- Fisher and `qp2` from the paper's centered FD at `ε=.075`;
- exact AtP baselines;
- `||Δ||/||r||` and alignment with receiver state;
- equal-norm full-dose KL;
- layerwise causal fidelity/gain/rewrite;
- output-logit causal fidelity/gain/rewrite.

The large trajectory tensors are stored as float16 memory maps in `/kaggle/working`; summary CSVs are checkpointed receiver-by-receiver.

In [ ]:
def run_large_model(model_key,spec):
    model,tok=load_large(spec)
    rows=tokenized_rows(tok,RAW_CONTEXTS,2*D+R)
    hosts,donors,receivers=rows[:D],rows[D:2*D],rows[2*D:2*D+R]
    n_layers=len(get_layers(model)); L=n_layers-spec['layer']; H=model.config.hidden_size
    print(model_key,'layers:',n_layers,'tracked suffix layers:',L,'hidden:',H)

    # ---- native hosts / donors / receivers ----
    host_u=[]; host_native_traj=[]; host_native_logits=[]
    donor_u=[]
    recv_native_traj=[]; recv_native_logits=[]

    for s in tqdm(range(0,D,PAIR_BATCH),desc=f'{model_key} native hosts'):
        batch=hosts[s:s+PAIR_BATCH]; ids=torch.stack([x['input_ids'] for x in batch])
        u,t,z=native_forward(model,ids,spec['layer'],True,True)
        host_u.append(u); host_native_traj.append(t); host_native_logits.append(z)
    for s in tqdm(range(0,D,PAIR_BATCH),desc=f'{model_key} native donors'):
        batch=donors[s:s+PAIR_BATCH]; ids=torch.stack([x['input_ids'] for x in batch])
        u,_,_=native_forward(model,ids,spec['layer'],True,False)
        donor_u.append(u)
    for s in tqdm(range(0,R,PAIR_BATCH),desc=f'{model_key} native receivers'):
        batch=receivers[s:s+PAIR_BATCH]; ids=torch.stack([x['input_ids'] for x in batch])
        _,t,z=native_forward(model,ids,spec['layer'],False,True)
        recv_native_traj.append(t); recv_native_logits.append(z)

    host_u=torch.cat(host_u); donor_u=torch.cat(donor_u)
    host_native_traj=torch.cat(host_native_traj); host_native_logits=torch.cat(host_native_logits)
    recv_native_traj=torch.cat(recv_native_traj); recv_native_logits=torch.cat(recv_native_logits)
    delta=(donor_u-host_u).float(); dnorm=delta.norm(dim=-1); target_norm=float(dnorm.median())
    delta_nm=delta*(target_norm/dnorm.clamp_min(1e-12))[:,None]
    recv_state=recv_native_traj[:,0,:].float(); recv_norm=recv_state.norm(dim=-1)

    # ---- exact AtP gradients: one suffix backward per receiver ----
    gtop=[]; gmargin=[]
    for k in tqdm(range(R),desc=f'{model_key} AtP gradients'):
        a,b=attribution_gradients(model,receivers[k]['input_ids'],spec['layer'])
        gtop.append(a); gmargin.append(b)
    gtop=torch.stack(gtop); gmargin=torch.stack(gmargin)

    # ---- host causal signatures ----
    host_sig_path=OUTDIR/f'{model_key}_host_signature.float16.mmap'
    host_sig=np.memmap(host_sig_path,mode='w+',dtype=np.float16,shape=(D,L,H))
    host_logit_path=OUTDIR/f'{model_key}_host_logit_signature.float16.mmap'
    V=host_native_logits.shape[-1]
    host_logit_sig=np.memmap(host_logit_path,mode='w+',dtype=np.float16,shape=(D,V))

    for s in tqdm(range(0,D,PAIR_BATCH),desc=f'{model_key} host causal signatures'):
        e=min(D,s+PAIR_BATCH); ids=torch.stack([x['input_ids'] for x in hosts[s:e]])
        t,z=patched_forward(model,ids,spec['layer'],delta[s:e],1.0,True)
        sig=t.float()-host_native_traj[s:e].float()
        host_sig[s:e]=sig.numpy().astype(np.float16)
        l=centered(z-host_native_logits[s:e])
        host_logit_sig[s:e]=l.numpy().astype(np.float16)
    host_sig.flush(); host_logit_sig.flush()

    # ---- receiver causal signatures + controls ----
    recv_sig_path=OUTDIR/f'{model_key}_receiver_signatures.float16.mmap'
    recv_sig=np.memmap(recv_sig_path,mode='w+',dtype=np.float16,shape=(D,R,L,H))
    row_csv=OUTDIR/f'{model_key}_camera_ready_rows_partial.csv'
    traj_csv=OUTDIR/f'{model_key}_RCCT_layerwise_partial.csv'
    rich_csv=OUTDIR/f'{model_key}_Richardson_partial.csv'
    rows_out=[]; traj_out=[]; rich_out=[]

    for k in range(R):
        rid=receivers[k]['input_ids']; z0=recv_native_logits[k:k+1]
        p=torch.softmax(z0.float(),dim=-1)[0]
        atp_top=(delta@gtop[k]).abs(); atp_margin=(delta@gmargin[k]).abs()

        for s in tqdm(range(0,D,PAIR_BATCH),desc=f'{model_key} receiver {k+1}/{R}',leave=False):
            e=min(D,s+PAIR_BATCH); B=e-s; d=delta[s:e]
            ids=rid[None,:].repeat(B,1)

            # Physical full-dose path: also supplies RCCT trajectories.
            t,zfull=patched_forward(model,ids,spec['layer'],d,1.0,True)
            rsig=t.float()-recv_native_traj[k:k+1].float()
            recv_sig[s:e,k]=rsig.numpy().astype(np.float16)
            kl=kl_native_patch(z0.expand(B,-1),zfull)

            # Local finite-difference response for paper's Fisher score + qp2.
            _,zp=patched_forward(model,ids,spec['layer'],d,FD_EPS,False)
            _,zm=patched_forward(model,ids,spec['layer'],d,-FD_EPS,False)
            v=(zp-zm)/(2*FD_EPS); pB=p[None,:].expand(B,-1)
            q=fisher_energy(v,pB); q2=qp2(v)

            # Physical equal-norm control.
            _,znm=patched_forward(model,ids,spec['layer'],delta_nm[s:e],1.0,False)
            klnm=kl_native_patch(z0.expand(B,-1),znm)

            # Full-vocabulary causal signature metrics, independent of Fisher.
            rlog=centered(zfull-z0.expand(B,-1))
            hlog=torch.from_numpy(np.asarray(host_logit_sig[s:e],dtype=np.float32))
            lf,lg,lr=causal_metrics(hlog,rlog)

            hsig=torch.from_numpy(np.asarray(host_sig[s:e],dtype=np.float32))
            tf,tg,tr=causal_metrics(hsig,rsig)

            for b,di in enumerate(range(s,e)):
                cosdr=float(torch.dot(delta[di],recv_state[k]) /
                            (delta[di].norm()*recv_state[k].norm()).clamp_min(1e-12))
                rows_out.append(dict(
                    model=model_key,layer=spec['layer'],displacement_id=di,receiver_id=k,
                    host_context_id=hosts[di]['context_id'],donor_context_id=donors[di]['context_id'],
                    receiver_context_id=receivers[k]['context_id'],
                    kl_full=float(kl[b]),Fisher=float(q[b]),qp2=float(q2[b]),
                    AtP_toplogp=float(atp_top[di]),AtP_margin=float(atp_margin[di]),
                    delta_norm=float(dnorm[di]),receiver_norm=float(recv_norm[k]),
                    relative_norm=float(dnorm[di]/recv_norm[k].clamp_min(1e-12)),
                    abs_cos_delta_receiver=abs(cosdr),
                    kl_equal_norm=float(klnm[b]),equal_norm_target=target_norm,
                    logit_fidelity=float(lf[b]),logit_gain=float(lg[b]),logit_rewrite=float(lr[b])
                ))
                for m in range(L):
                    traj_out.append(dict(model=model_key,displacement_id=di,receiver_id=k,
                                         downstream_layer=spec['layer']+m,
                                         fidelity=float(tf[b,m]),gain=float(tg[b,m]),rewrite=float(tr[b,m])))

                # Same-model numerical convergence on a small frozen subset.
                if len(rich_out)<RICHARDSON_N:
                    h=FD_EPS/2
                    _,zph=patched_forward(model,rid,spec['layer'],d[b:b+1],h,False)
                    _,zmh=patched_forward(model,rid,spec['layer'],d[b:b+1],-h,False)
                    vh=(zph-zmh)/(2*h); vr=(4*vh-v[b:b+1])/3
                    qe=float(fisher_energy(v[b:b+1],p[None,:])[0])
                    qh=float(fisher_energy(vh,p[None,:])[0]); qr=float(fisher_energy(vr,p[None,:])[0])
                    rich_out.append(dict(model=model_key,pair_id=len(rich_out),displacement_id=di,receiver_id=k,
                                         cos_eps_half=float(cosine_rows(v[b:b+1],vh)[0]),
                                         cos_eps_rich=float(cosine_rows(v[b:b+1],vr)[0]),
                                         rel_vec_eps_vs_rich=float((v[b]-vr[0]).norm()/vr[0].norm().clamp_min(1e-12)),
                                         rel_Fisher_eps_vs_rich=abs(qe-qr)/max(abs(qr),1e-30),
                                         Fisher_eps=qe,Fisher_half=qh,Fisher_richardson=qr))

        recv_sig.flush()
        atomic_csv(pd.DataFrame(rows_out),row_csv)
        atomic_csv(pd.DataFrame(traj_out),traj_csv)
        atomic_csv(pd.DataFrame(rich_out),rich_csv)
        print('checkpointed receiver',k,'for',model_key)

    rows_df=pd.DataFrame(rows_out); traj_df=pd.DataFrame(traj_out); rich_df=pd.DataFrame(rich_out)
    atomic_csv(rows_df,OUTDIR/f'{model_key}_camera_ready_rows.csv')
    atomic_csv(traj_df,OUTDIR/f'{model_key}_RCCT_layerwise.csv')
    atomic_csv(rich_df,OUTDIR/f'{model_key}_Richardson.csv')

    # Sanity check: target-layer causal vectors should begin nearly identical.
    first=traj_df[traj_df.downstream_layer==spec['layer']].fidelity
    print(model_key,'median fidelity at injection layer:',float(first.median()))
    if first.median()<0.95: print('WARNING: injection-layer fidelity unexpectedly low; inspect hooks before paper use.')

    # Save IDs and displacement metadata.
    atomic_csv(pd.DataFrame({
        'displacement_id':range(D),'host_context_id':[x['context_id'] for x in hosts],
        'donor_context_id':[x['context_id'] for x in donors],'delta_norm':dnorm.numpy()}),
        OUTDIR/f'{model_key}_displacements.csv')
    atomic_csv(pd.DataFrame({'receiver_id':range(R),'receiver_context_id':[x['context_id'] for x in receivers],
                             'receiver_norm':recv_norm.numpy()}),OUTDIR/f'{model_key}_receivers.csv')

    # Return paths; transformation analysis runs on CPU after model is unloaded.
    meta=dict(L=L,H=H,target_layer=spec['layer'],host_sig_path=str(host_sig_path),recv_sig_path=str(recv_sig_path))
    del tok,host_u,donor_u,host_native_traj,host_native_logits,recv_native_traj,recv_native_logits
    cleanup(model)
    return rows_df,traj_df,rich_df,meta

## 4. Canonical summary statistics

This prevents the public-paper from mixing different estimands. On the **same row population** we report:

- pooled/global Spearman — descriptive;
- median within-fixed-displacement Spearman — primary crossed statistic;
- displacement + receiver residual rank correlation — pair-specific association beyond generic receiver fragility.

In [ ]:
def safe_spearman(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float); ok=np.isfinite(x)&np.isfinite(y)
    if ok.sum()<3 or np.std(x[ok])==0 or np.std(y[ok])==0: return np.nan
    return float(spearmanr(x[ok],y[ok]).statistic)

def safe_pearson(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float); ok=np.isfinite(x)&np.isfinite(y)
    if ok.sum()<3 or np.std(x[ok])==0 or np.std(y[ok])==0: return np.nan
    return float(pearsonr(x[ok],y[ok]).statistic)

def rank_residual(values,dids,rids,iters=100):
    z=rankdata(np.asarray(values,float),method='average').astype(float); z-=z.mean()
    dids=np.asarray(dids); rids=np.asarray(rids)
    for _ in range(iters):
        old=z.copy()
        for g in np.unique(dids): m=dids==g; z[m]-=z[m].mean()
        for g in np.unique(rids): m=rids==g; z[m]-=z[m].mean()
        if np.max(np.abs(z-old))<1e-12: break
    return z

def residual_corr(df,pred,outcome):
    g=df[np.isfinite(df[pred])&np.isfinite(df[outcome])]
    if len(g)<10:return np.nan
    return safe_pearson(rank_residual(g[pred],g.displacement_id,g.receiver_id),
                        rank_residual(g[outcome],g.displacement_id,g.receiver_id))

def within_disp(df,pred,outcome):
    vals=[]
    for _,g in df.groupby('displacement_id'):
        r=safe_spearman(g[pred],g[outcome])
        if np.isfinite(r): vals.append(r)
    if not vals:return np.nan,np.nan,0
    vals=np.asarray(vals); return float(np.median(vals)),float((vals>0).mean()),len(vals)

def canonical_table(df,model_key):
    preds=['Fisher','qp2','AtP_toplogp','AtP_margin','receiver_norm','relative_norm','abs_cos_delta_receiver']
    rows=[]
    for pred in preds:
        med,pos,n=within_disp(df,pred,'kl_full')
        rows.append(dict(model=model_key,predictor=pred,N_rows=len(df),N_displacements=D,N_receivers=R,
                         global_spearman=safe_spearman(df[pred],df.kl_full),
                         median_within_displacement_spearman=med,
                         fraction_displacements_positive=pos,
                         residual_rank_corr=residual_corr(df,pred,'kl_full')))
    out=pd.DataFrame(rows); atomic_csv(out,OUTDIR/f'{model_key}_PREDICTOR_SUMMARY.csv')
    return out

## 5. Held-out receiver-transformation test (the new result)

A plain fidelity curve only says that causal effects diverge. The stronger test asks whether the divergence has **receiver-specific structure that generalizes to unseen displacements**.

At each downstream layer, fit on 75% of displacements and test on the untouched 25%:

- **Identity:** receiver does nothing systematic;
- **Gain-only:** receiver only amplifies/attenuates;
- **Global map:** all receivers share one transformation;
- **Receiver map:** each receiver gets its own low-rank map;
- **Wrong-receiver map:** use another receiver's map as a negative control.

To avoid fitting a 4096×4096 matrix from 72 examples, all linear maps are fit in a rank-16 PCA subspace learned **only from training displacements**.

In [ ]:
def cosine_np(a,b,eps=1e-12):
    num=(a*b).sum(1); den=np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)
    return num/np.maximum(den,eps)

def nrmse_np(pred,y):
    return np.linalg.norm(pred-y,axis=1)/np.maximum(np.linalg.norm(y,axis=1),1e-12)

def run_map_benchmark(model_key,meta):
    L,H=meta['L'],meta['H']
    Xmm=np.memmap(meta['host_sig_path'],mode='r',dtype=np.float16,shape=(D,L,H))
    Ymm=np.memmap(meta['recv_sig_path'],mode='r',dtype=np.float16,shape=(D,R,L,H))

    rng=np.random.default_rng(SEED)
    perm=rng.permutation(D); ntr=int(round(TRAIN_FRAC*D)); train=perm[:ntr]; test=perm[ntr:]
    pd.DataFrame({'displacement_id':np.arange(D),'split':['train' if i in set(train) else 'test' for i in range(D)]}).to_csv(
        OUTDIR/f'{model_key}_RCCT_train_test_split.csv',index=False)

    rec=[]
    for m in tqdm(range(L),desc=f'{model_key} held-out receiver maps'):
        X=np.asarray(Xmm[:,m,:],dtype=np.float32)
        Y=np.asarray(Ymm[:,:,m,:],dtype=np.float32) # D,R,H

        # PCA basis sees training displacements only, but all receiver outcomes for those training displacements.
        pool=np.concatenate([X[train],Y[train].reshape(-1,H)],axis=0)
        r=min(MAP_RANK,pool.shape[0]-1,H)
        pca=PCA(n_components=r,svd_solver='randomized',random_state=SEED).fit(pool)
        XtrZ=pca.transform(X[train]); XteZ=pca.transform(X[test])
        YtrZ=[pca.transform(Y[train,k]) for k in range(R)]

        # One global map across all receiver training outcomes.
        Xg=np.tile(XtrZ,(R,1)); Yg=np.concatenate(YtrZ,axis=0)
        global_map=Ridge(alpha=RIDGE_ALPHA,fit_intercept=True).fit(Xg,Yg)

        receiver_maps=[]; gains=[]
        for k in range(R):
            receiver_maps.append(Ridge(alpha=RIDGE_ALPHA,fit_intercept=True).fit(XtrZ,YtrZ[k]))
            gains.append(float((X[train]*Y[train,k]).sum()/max((X[train]*X[train]).sum(),1e-12)))

        for k in range(R):
            y=Y[test,k]
            preds={
                'identity':X[test],
                'gain_only':gains[k]*X[test],
                'global_map':pca.inverse_transform(global_map.predict(XteZ)),
                'receiver_map':pca.inverse_transform(receiver_maps[k].predict(XteZ)),
                'wrong_receiver_map':pca.inverse_transform(receiver_maps[(k+1)%R].predict(XteZ)),
            }
            for method,pred in preds.items():
                cs=cosine_np(pred,y); er=nrmse_np(pred,y)
                rec.append(dict(model=model_key,downstream_layer=meta['target_layer']+m,receiver_id=k,
                                method=method,N_train=len(train),N_test=len(test),pca_rank=r,
                                mean_cosine=float(np.mean(cs)),median_cosine=float(np.median(cs)),
                                mean_nrmse=float(np.mean(er)),median_nrmse=float(np.median(er))))

    out=pd.DataFrame(rec); atomic_csv(out,OUTDIR/f'{model_key}_RCCT_HELDOUT_MAP_BENCHMARK.csv')
    summary=out.groupby(['model','downstream_layer','method'],as_index=False).agg(
        mean_cosine=('mean_cosine','mean'),median_cosine=('median_cosine','median'),
        mean_nrmse=('mean_nrmse','mean'),median_nrmse=('median_nrmse','median'))
    atomic_csv(summary,OUTDIR/f'{model_key}_RCCT_HELDOUT_MAP_SUMMARY.csv')

    fig,ax=plt.subplots(figsize=(7.2,4.6))
    for method,g in summary.groupby('method'):
        ax.plot(g.downstream_layer,g.mean_cosine,marker='o',label=method)
    ax.set_xlabel('Downstream layer'); ax.set_ylabel('Held-out cosine prediction')
    ax.set_title(f'{model_key}: receiver transformation generalization')
    ax.legend(fontsize=8); fig.tight_layout()
    fig.savefig(OUTDIR/f'{model_key}_RCCT_heldout_map.png',dpi=220); plt.close(fig)
    return out,summary

## 6. Run Qwen3-8B and Mistral-7B sequentially

This is the main GPU section. The notebook unloads each 7B–8B model before CPU map fitting / loading the next one.

In [ ]:
notify('MLP public-paper run STARTED','T4x2 final alternative controls + RCCT started.')
ALL_ROWS=[]; ALL_TRAJ=[]; ALL_RICH=[]; ALL_CANON=[]; ALL_MAP=[]; ALL_MAP_SUM=[]

try:
    for model_key,spec in MODEL_SPECS.items():
        print('\n'+'='*100+'\n',model_key,'\n'+'='*100)
        rows_df,traj_df,rich_df,meta=run_large_model(model_key,spec)
        can=canonical_table(rows_df,model_key)
        maps,mapsum=run_map_benchmark(model_key,meta)
        ALL_ROWS.append(rows_df); ALL_TRAJ.append(traj_df); ALL_RICH.append(rich_df)
        ALL_CANON.append(can); ALL_MAP.append(maps); ALL_MAP_SUM.append(mapsum)

    ALL_ROWS=pd.concat(ALL_ROWS,ignore_index=True); atomic_csv(ALL_ROWS,OUTDIR/'ALL_MODELS_camera_ready_rows.csv')
    ALL_TRAJ=pd.concat(ALL_TRAJ,ignore_index=True); atomic_csv(ALL_TRAJ,OUTDIR/'ALL_MODELS_RCCT_layerwise.csv')
    ALL_RICH=pd.concat(ALL_RICH,ignore_index=True); atomic_csv(ALL_RICH,OUTDIR/'ALL_MODELS_Richardson.csv')
    ALL_CANON=pd.concat(ALL_CANON,ignore_index=True); atomic_csv(ALL_CANON,OUTDIR/'ALL_MODELS_PREDICTOR_SUMMARY.csv')
    ALL_MAP=pd.concat(ALL_MAP,ignore_index=True); atomic_csv(ALL_MAP,OUTDIR/'ALL_MODELS_RCCT_HELDOUT_MAP_BENCHMARK.csv')
    ALL_MAP_SUM=pd.concat(ALL_MAP_SUM,ignore_index=True); atomic_csv(ALL_MAP_SUM,OUTDIR/'ALL_MODELS_RCCT_HELDOUT_MAP_SUMMARY.csv')

    display(ALL_CANON)
    display(ALL_MAP_SUM.groupby(['model','method'],as_index=False).agg(mean_cosine=('mean_cosine','mean'),mean_nrmse=('mean_nrmse','mean')))
except Exception as e:
    notify('MLP public-paper run FAILED',f'{type(e).__name__}: {str(e)[:300]}','urgent')
    raise

## 7. Full-FP32 exact-JVP closure on six Pythia-70M runs

A T4×2 session cannot honestly do a full-Qwen/Mistral FP32 exact JVP. Instead:

- the headline models get same-model `ε/2` + Richardson convergence;
- the already-used six independent Pythia runs get a literal **full-FP32 exact autodiff JVP** audit.

This should be described precisely in the public-paper rather than overstated.

In [ ]:
def pythia_exact_jvp(model,ids,layer_idx,delta_cpu):
    dev=input_device(model)
    alpha=torch.zeros((),device=dev,dtype=torch.float32,requires_grad=True)
    tangent=torch.ones_like(alpha)

    def f(a):
        captured={}; mlp=get_mlp(model,layer_idx)
        def hook(module,inp,out):
            d=delta_cpu.to(out.device,dtype=out.dtype)
            add=torch.zeros_like(out); add[:,-1,:]=d
            return out+a.to(out.device,dtype=out.dtype)*add
        h=mlp.register_forward_hook(hook)
        try:
            return model(input_ids=ids[None,:].to(dev),use_cache=False,return_dict=True).logits[0,-1,:]
        finally: h.remove()

    with torch.enable_grad():
        y,v=torch.autograd.functional.jvp(f,alpha,tangent,create_graph=False,strict=False)
    return y.detach().float().cpu(),v.detach().float().cpu()

@torch.inference_mode()
def pythia_fd(model,ids,layer_idx,delta,eps):
    _,zp=patched_forward(model,ids,layer_idx,delta[None,:],eps,False)
    _,zm=patched_forward(model,ids,layer_idx,delta[None,:],-eps,False)
    return (zp-zm)[0]/(2*eps)

def run_pythia_closure(run_name,model_id):
    tok=AutoTokenizer.from_pretrained(model_id,use_fast=True)
    if tok.pad_token_id is None: tok.pad_token_id=tok.eos_token_id
    model=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=torch.float32,device_map={'':0},low_cpu_mem_usage=True)
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)

    need=2*PYTHIA_PAIRS_PER_RUN+PYTHIA_PAIRS_PER_RUN
    rows=tokenized_rows(tok,RAW_CONTEXTS,need)
    hosts=rows[:PYTHIA_PAIRS_PER_RUN]; donors=rows[PYTHIA_PAIRS_PER_RUN:2*PYTHIA_PAIRS_PER_RUN]
    recvs=rows[2*PYTHIA_PAIRS_PER_RUN:]
    hu=[]; du=[]
    for x in hosts:
        u,_,_=native_forward(model,x['input_ids'][None,:],PYTHIA_LAYER,True,False); hu.append(u[0])
    for x in donors:
        u,_,_=native_forward(model,x['input_ids'][None,:],PYTHIA_LAYER,True,False); du.append(u[0])
    delta=torch.stack(du)-torch.stack(hu)

    out=[]
    for i in tqdm(range(PYTHIA_PAIRS_PER_RUN),desc=f'Pythia {run_name} FP32 JVP'):
        rid=recvs[i]['input_ids']; d=delta[i].float()
        z0,vex=pythia_exact_jvp(model,rid,PYTHIA_LAYER,d)
        vfd=pythia_fd(model,rid,PYTHIA_LAYER,d,FD_EPS)
        p=torch.softmax(z0.float(),dim=-1)
        qex=float(fisher_energy(vex[None,:],p[None,:])[0]); qfd=float(fisher_energy(vfd[None,:],p[None,:])[0])
        out.append(dict(run=run_name,pair_id=i,
                        cos_exact_FD=float(cosine_rows(vex[None,:],vfd[None,:])[0]),
                        rel_vector_error=float((vfd-vex).norm()/vex.norm().clamp_min(1e-12)),
                        rel_Fisher_error=abs(qfd-qex)/max(abs(qex),1e-30),Fisher_exact=qex,Fisher_FD=qfd))
    df=pd.DataFrame(out); atomic_csv(df,OUTDIR/f'pythia_{run_name}_FP32_exact_JVP.csv')
    del tok; cleanup(model); return df

PY=[]
for run_name,model_id in PYTHIA_MODELS.items():
    try: PY.append(run_pythia_closure(run_name,model_id))
    except Exception as e:
        print('Pythia closure failed for',run_name,repr(e))
        (OUTDIR/f'pythia_{run_name}_ERROR.txt').write_text(traceback.format_exc())

PY_ALL=pd.concat(PY,ignore_index=True); atomic_csv(PY_ALL,OUTDIR/'PYTHIA_ALL_FP32_exact_JVP.csv')
PY_SUM=PY_ALL.groupby('run',as_index=False).agg(N=('pair_id','count'),
    median_cos_exact_FD=('cos_exact_FD','median'),median_rel_vector_error=('rel_vector_error','median'),
    median_rel_Fisher_error=('rel_Fisher_error','median'))
atomic_csv(PY_SUM,OUTDIR/'PYTHIA_FP32_exact_JVP_SUMMARY.csv')
display(PY_SUM)

## 8. Final decision table — what the new result actually supports

The notebook does **not** automatically declare the hypothesis true. It summarizes the held-out comparison so the paper can be written according to the data:

- `receiver_map > global_map` and `receiver_map > wrong_receiver_map` consistently → evidence for a reusable receiver-specific transformation;
- fidelity/gain differ by receiver but held-out receiver maps do not win → systematic receiver modulation, but largely update-specific;
- little difference at all → RCCT hypothesis not supported; keep the alternative controls and do not force the new claim.

In [ ]:
# Compact decision table across downstream layers/models.
DEC=[]
for (model,layer),g in ALL_MAP_SUM.groupby(['model','downstream_layer']):
    vals={r.method:r.mean_cosine for _,r in g.iterrows()}
    if {'receiver_map','global_map','wrong_receiver_map','identity','gain_only'} <= set(vals):
        DEC.append(dict(model=model,downstream_layer=layer,
                        receiver_map=vals['receiver_map'],global_map=vals['global_map'],
                        wrong_receiver_map=vals['wrong_receiver_map'],identity=vals['identity'],gain_only=vals['gain_only'],
                        receiver_minus_global=vals['receiver_map']-vals['global_map'],
                        receiver_minus_wrong=vals['receiver_map']-vals['wrong_receiver_map']))
DEC=pd.DataFrame(DEC); atomic_csv(DEC,OUTDIR/'RCCT_DECISION_TABLE.csv')
display(DEC)

# Alternative-control compact summary.
print('\nCanonical alternative controls:')
display(ALL_CANON.sort_values(['model','median_within_displacement_spearman'],ascending=[True,False]))
print('\nLarge-model finite-difference convergence:')
display(ALL_RICH.groupby('model')[['cos_eps_half','cos_eps_rich','rel_vec_eps_vs_rich','rel_Fisher_eps_vs_rich']].median())

## 9. Package the final artifact

Large temporary memory maps are excluded from the ZIP; all analysis tables, train/test split, alternative controls, RCCT summaries, figures, and numerical audits are retained.

**After completion, use Kaggle `Save Version` so `/kaggle/working` outputs persist.**

In [ ]:
# Save context hashes only, not raw C4 text.
pd.DataFrame([{'context_id':x['context_id']} for x in RAW_CONTEXTS]).to_csv(OUTDIR/'final control_context_ids.csv',index=False)

final_zip=Path('/kaggle/working/MLP_Transplants_FINAL_CAMERA_READY_RESULTS.zip')
if final_zip.exists(): final_zip.unlink()
with zipfile.ZipFile(final_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTDIR.rglob('*')):
        if not p.is_file(): continue
        if p.suffix=='.mmap' or '_partial.csv' in p.name: continue
        z.write(p,p.relative_to(OUTDIR))
print('FINAL ZIP:',final_zip,'size MiB:',round(final_zip.stat().st_size/2**20,2))
notify('MLP public-paper run COMPLETE',f'Final results: {final_zip.name}','high')